# Notebook 08 — Etiquetado del conjunto de datos

En esta fase se construye la variable objetivo que será utilizada durante el entrenamiento del modelo de clasificación. Dado que no existe una base de datos histórica homogénea que identifique de forma explícita los episodios de inundación para cada estación y fecha, se define una etiqueta de riesgo a partir de reglas hidrometeorológicas basadas en precipitación diaria, precipitación acumulada y condiciones de saturación previa del terreno.

El objetivo es generar una variable binaria capaz de diferenciar situaciones potencialmente peligrosas de condiciones meteorológicas ordinarias, proporcionando así una base consistente para el entrenamiento supervisado del modelo.

## 1. Análisis exploratorio de umbrales

Antes de definir la etiqueta de riesgo se analiza la distribución estadística de las variables de precipitación. Este análisis permite identificar valores extremos y seleccionar umbrales coherentes con la frecuencia observada de los episodios intensos dentro del conjunto de datos.

In [1]:
import pandas as pd

dataset = pd.read_parquet("../data/processed/features_modelo.parquet")

# Percentiles de las variables de lluvia
print("=== Distribución de precip_dia (mm) ===")
for p in [50, 75, 90, 95, 98, 99, 99.5]:
    print(f"  Percentil {p}: {dataset['precip_dia'].quantile(p/100):.1f} mm")

print("\n=== Distribución de precip_3d (mm) ===")
for p in [50, 75, 90, 95, 98, 99, 99.5]:
    print(f"  Percentil {p}: {dataset['precip_3d'].quantile(p/100):.1f} mm")

print("\n=== Distribución de precip_previa_7d (mm) ===")
for p in [50, 75, 90, 95, 98, 99]:
    print(f"  Percentil {p}: {dataset['precip_previa_7d'].quantile(p/100):.1f} mm")

# Cuántas filas superarían cada umbral candidato
print("\n=== Simulación de la etiqueta ===")
for umbral in [50, 80, 100, 120]:
    n = (dataset['precip_dia'] >= umbral).sum()
    print(f"  precip_dia >= {umbral} mm: {n} filas ({n/len(dataset)*100:.1f}%)")

=== Distribución de precip_dia (mm) ===
  Percentil 50: 2.8 mm
  Percentil 75: 10.0 mm
  Percentil 90: 21.6 mm
  Percentil 95: 33.4 mm
  Percentil 98: 53.0 mm
  Percentil 99: 72.3 mm
  Percentil 99.5: 97.8 mm

=== Distribución de precip_3d (mm) ===
  Percentil 50: 5.0 mm
  Percentil 75: 16.6 mm
  Percentil 90: 36.8 mm
  Percentil 95: 59.6 mm
  Percentil 98: 96.4 mm
  Percentil 99: 134.2 mm
  Percentil 99.5: 185.9 mm

=== Distribución de precip_previa_7d (mm) ===
  Percentil 50: 0.8 mm
  Percentil 75: 14.0 mm
  Percentil 90: 39.8 mm
  Percentil 95: 64.0 mm
  Percentil 98: 106.8 mm
  Percentil 99: 147.8 mm

=== Simulación de la etiqueta ===
  precip_dia >= 50 mm: 447 filas (2.3%)
  precip_dia >= 80 mm: 150 filas (0.8%)
  precip_dia >= 100 mm: 93 filas (0.5%)
  precip_dia >= 120 mm: 60 filas (0.3%)


## 2. Construcción de la etiqueta de riesgo

La etiqueta se construye mediante una combinación de reglas que representan distintos mecanismos asociados a la generación de inundaciones:

- Precipitación intensa en un único día.
- Acumulación significativa de precipitación durante varios días consecutivos.
- Episodios de lluvia sobre terrenos previamente saturados.

Una observación se clasifica como riesgo cuando cumple al menos una de estas condiciones.

In [2]:
import pandas as pd

dataset = pd.read_parquet("../data/processed/features_modelo.parquet")

# Definimos la etiqueta según la regla calibrada
cond_intensa   = dataset["precip_dia"] >= 60
cond_acumulada = dataset["precip_3d"] >= 100
cond_saturado  = (dataset["precip_dia"] >= 40) & (dataset["precip_previa_7d"] >= 40)

dataset["riesgo"] = (cond_intensa | cond_acumulada | cond_saturado).astype(int)

# ¿Cómo queda el balance de clases?
n_riesgo = dataset["riesgo"].sum()
n_total = len(dataset)
print(f"Filas de RIESGO (1):  {n_riesgo} ({n_riesgo/n_total*100:.1f}%)")
print(f"Filas NORMALES (0):   {n_total - n_riesgo} ({(n_total-n_riesgo)/n_total*100:.1f}%)")

# Desglose de qué condición dispara cada caso
print(f"\nDesglose de las condiciones:")
print(f"  Por lluvia intensa (>=60mm/día):  {cond_intensa.sum()}")
print(f"  Por acumulado 3d (>=100mm):       {cond_acumulada.sum()}")
print(f"  Por suelo saturado:               {cond_saturado.sum()}")

Filas de RIESGO (1):  590 (3.0%)
Filas NORMALES (0):   19157 (97.0%)

Desglose de las condiciones:
  Por lluvia intensa (>=60mm/día):  311
  Por acumulado 3d (>=100mm):       374
  Por suelo saturado:               169


590 filas de riesgo (3.0%) — está justo en la franja que buscábamos (3-8%). Ni demasiado raro (el modelo tendría con qué aprender) ni demasiado común. Es un balance trabajable.
El desglose tiene sentido físico:

- 311 casos por lluvia intensa puntual
- 374 por acumulado de 3 días
- 169 por suelo saturado

(suman más de 590 porque una fila puede cumplir varias condiciones a la vez — normal)

## 3. Verificación sobre el episodio DANA de octubre de 2024

Como comprobación cualitativa se analiza el comportamiento de la etiqueta durante el día de máxima intensidad del episodio DANA. Se verifica que estaciones especialmente afectadas, como Chiva, Forata o la Rambla del Poyo, queden correctamente clasificadas como situaciones de riesgo.

In [3]:
# ¿El 29 de octubre, Chiva/Forata/Poyo están etiquetadas como riesgo?
dia_dana = dataset[dataset["fecha"] == "2024-10-29"]

print("=== Etiquetas del 29 de octubre 2024 ===")
for clave in ["CHIVA", "FORATA", "POYO", "UTIEL", "TOUS", "ANTELLA"]:
    filas = dia_dana[dia_dana["descripcion"].str.contains(clave, case=False, na=False)]
    for _, row in filas.iterrows():
        etiqueta = "RIESGO" if row["riesgo"] == 1 else "normal"
        print(f"  [{etiqueta:6s}] {row['descripcion']:50s} "
              f"precip_dia={row['precip_dia']:.0f}mm")

=== Etiquetas del 29 de octubre 2024 ===
  [RIESGO] Pluviometro de Chiva Chiva                         precip_dia=461mm
  [RIESGO] Embalse de Forata Yátova                           precip_dia=188mm
  [RIESGO] Marco en rambla del Poyo Riba-roja                 precip_dia=213mm
  [RIESGO] Embalse de Tous Tous                               precip_dia=53mm
  [normal] Azud de Antella Antella                            precip_dia=23mm


Chiva (461 mm), Forata (188), Poyo (213) → todas RIESGO.

Las tres estaciones del desastre están bien etiquetadas.

Antella sale "normal" con 23 mm. ¿Es un error? No, es correcto y además es revelador. El 29 de octubre en Antella (sur de la cuenca) llovió poco. Antella sufrió la inundación días después, cuando le llegó el agua del Júcar desde aguas arriba. Esto ilustra perfectamente lo que mostró del MDT: la lluvia y el daño no coinciden en el espacio ni en el tiempo.

La etiqueta está capturando "dónde llovió fuerte", no "dónde se inundó" — y eso es exactamente lo correcto para esta primera versión del modelo.

## 4. Exportación del dataset etiquetado

Una vez generada y validada la variable objetivo, el conjunto de datos se almacena para su utilización en la fase de entrenamiento y evaluación de los modelos predictivos.

In [4]:
# Guardamos el dataset con la etiqueta incluida
out_path = "../data/processed/dataset_etiquetado.parquet"
dataset.to_parquet(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"  {len(dataset):,} filas, {dataset['riesgo'].sum()} de riesgo")

Guardado: ../data/processed/dataset_etiquetado.parquet
  19,747 filas, 590 de riesgo
